<a href="https://colab.research.google.com/github/RyanHadiA/Skripsi-TA/blob/main/Clear%20model_deep_learning_Skripsi(TA)Biner_(Model_BERT_Gabungan_Seluruh_Dataset).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load & Split data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import re

file_path = '/content/drive/MyDrive/Combined All Dataset/Teks panjang-pendek(192)/preprocessed_Gabungan Seluruh Dataset_long6046.csv'
#file_path = '/content/drive/MyDrive/Combined All Dataset/Teks panjang-pendek(192)/preprocessed_Gabungan Seluruh Dataset_short31584.csv'
df = pd.read_csv(file_path)

# Hitung total jumlah data
total_samples = len(df)
print(f"Total data: {total_samples}\n")

# ==== Split Dataset =====
# Mengacak dataset terlebih dahulu
df = shuffle(df, random_state=42)

# Split: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print jumlah tiap subset
print(f"Jumlah data pelatihan: {len(train_df)}")
print(f"Jumlah data validasi: {len(val_df)}")
print(f"Jumlah data pengujian: {len(test_df)}")

Tokenizer BERT

In [ ]:
from transformers import BertTokenizerFast
import torch
from torch.utils.data import Dataset
from typing import Optional, Union
import pandas as pd
import os
import random
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')


class ConfigTM :
    MAX_LENGTH = 256
    HIDDEN_DIM = 256


class PersonalityDataset(Dataset):
    def __init__(
        self,
        data: Union[str, pd.DataFrame],
        tokenizer: BertTokenizerFast,
        max_len: int = ConfigTM.MAX_LENGTH,
        text_col: str = "Text",
        label_cols: Optional[list] = None
    ):

        if isinstance(data, str):
            if not os.path.isfile(data):
                raise FileNotFoundError(f"File '{data}' tidak ditemukan.")
            df = pd.read_csv(data)
        elif isinstance(data, pd.DataFrame):
            df = data.copy()
        else:
            raise ValueError("Argumen `data` harus str (path) atau pd.DataFrame.")

        if text_col not in df.columns:
            raise ValueError(f"Kolom '{text_col}' tidak ditemukan di data.")
        if label_cols is None:
            label_cols = df.columns[-5:].tolist()
        for lab in label_cols:
            if lab not in df.columns:
                raise ValueError(f"Kolom label '{lab}' tidak ditemukan di data.")

        self.tokenizer = tokenizer
        self.max_len = max_len

        self.texts = df[text_col].astype(str).tolist()
        self.labels = df[label_cols].astype(float).values

        encodings = tokenizer(
            self.texts,
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        self.input_ids = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']

        self.labels_tensor = torch.tensor(self.labels, dtype=torch.float)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels_tensor[idx]
        }


train_dataset = PersonalityDataset(train_df, tokenizer, max_len=ConfigTM.MAX_LENGTH)
val_dataset   = PersonalityDataset(val_df,   tokenizer, max_len=ConfigTM.MAX_LENGTH)
test_dataset  = PersonalityDataset(test_df,  tokenizer, max_len=ConfigTM.MAX_LENGTH)


print("\nContoh hasil preprocessing 1 sample:")
sample = train_dataset[0]
print(f"Input IDs    : {sample['input_ids'].shape}  # tensor of length {sample['input_ids'].shape[0]}")
print(f"Attention Mask: {sample['attention_mask'].shape}")
print(f"Labels (OCEAN): {sample['labels']}  # shape {sample['labels'].shape}")

Definisi Model

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class SimpleBertOcean(nn.Module):
    def __init__(
        self,
        pretrained_model_name: str = 'bert-base-uncased',
        hidden_dim: int = ConfigTM.HIDDEN_DIM,
        dropout: float = 0.3,
        num_labels: int = 5,
        freeze_bert: bool = True
    ):
        super(SimpleBertOcean, self).__init__()

        self.bert = BertModel.from_pretrained(pretrained_model_name)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output

        logits = self.classifier(pooled)
        return logits

model = SimpleBertOcean(
    pretrained_model_name='bert-base-uncased',
    hidden_dim=ConfigTM.HIDDEN_DIM,
    dropout=0.3,
    num_labels=5,
    freeze_bert=True
)

print(model)

Hyperparameter dan pelatihan

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW, SGD
import matplotlib.pyplot as plt
from google.colab import drive

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

class Config:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    LEARNING_RATE = 3e-5
    BATCH_SIZE    = 32
    EPOCHS        = 3

    OPTIMIZER = 'adamw'

    LOSS_FN = 'bce_logits'

    PIN_MEMORY  = True
    NUM_WORKERS = 2

    CHECKPOINT_PATH = f'/content/drive/MyDrive/model_checkpoints/v1-tambahan metode threshold/bert_cekpt_Gabungan Seluruh Dataset_biner([{LEARNING_RATE}]-{BATCH_SIZE}-{EPOCHS}-{NUM_WORKERS})MX{ConfigTM.MAX_LENGTH} H{ConfigTM.HIDDEN_DIM} JMLH{len(df)}.pth'


train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    pin_memory=Config.PIN_MEMORY,
    num_workers=Config.NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    pin_memory=Config.PIN_MEMORY,
    num_workers=Config.NUM_WORKERS
)


model = SimpleBertOcean(
    pretrained_model_name='bert-base-uncased',
    hidden_dim=ConfigTM.HIDDEN_DIM,
    dropout=0.3,
    num_labels=5,
    freeze_bert=True
).to(Config.DEVICE)

if Config.LOSS_FN.lower() == 'bce':
    loss_fn = nn.BCELoss()
else:
    loss_fn = nn.BCEWithLogitsLoss()

trainable_params = filter(lambda p: p.requires_grad, model.parameters())
if Config.OPTIMIZER.lower() == 'sgd':
    optimizer = SGD(trainable_params, lr=Config.LEARNING_RATE)
else:
    optimizer = AdamW(trainable_params, lr=Config.LEARNING_RATE)


def train_epoch(loader, model, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss   = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def eval_epoch(loader, model, loss_fn, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss   = loss_fn(logits, labels)
            running_loss += loss.item()

    return running_loss / len(loader)


train_losses, val_losses = [], []
start_all = time.time()

for epoch in range(1, Config.EPOCHS + 1):
    train_loss = train_epoch(train_loader, model, loss_fn, optimizer, Config.DEVICE)
    val_loss   = eval_epoch(val_loader,   model, loss_fn, Config.DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch}/{Config.EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

total_secs     = time.time() - start_all
total_time_str = time.strftime("%H:%M:%S", time.gmtime(total_secs))
print(f"\nTotal waktu pelatihan: {total_time_str}\n")

plt.figure(figsize=(8,5))
plt.plot(range(1, Config.EPOCHS+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, Config.EPOCHS+1), val_losses,   label='Val Loss',   marker='s')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Loss Curve (Train vs Val)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Simpan pelatihan sebelum ke tahap selanjutnya

In [ ]:
import torch
import os

os.makedirs(os.path.dirname(Config.CHECKPOINT_PATH), exist_ok=True)

checkpoint = {
    'dataset_info': {
        'total_data': len(df),
        'train_size': len(train_dataset),
        'val_size'  : len(val_dataset),
        'test_size' : len(test_dataset)
    },
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'hyperparameters': {
        'learning_rate': Config.LEARNING_RATE,
        'batch_size'   : Config.BATCH_SIZE,
        'epochs'       : Config.EPOCHS,
        'optimizer'    : Config.OPTIMIZER,
        'loss_fn'      : Config.LOSS_FN,
        'pin_memory'   : Config.PIN_MEMORY,
        'num_workers'  : Config.NUM_WORKERS,
        'max_length'   : ConfigTM.MAX_LENGTH,
        'freeze_bert'  : True,
        'pretrained_model_name': 'bert-base-uncased',
        'hidden_dim'   : ConfigTM.HIDDEN_DIM,
        'dropout'      : 0.3,
        'num_labels'   : 5
    },
    'train_losses'  : train_losses,
    'val_losses'    : val_losses,
    'training_time' : total_time_str
}

torch.save(checkpoint, Config.CHECKPOINT_PATH)
print(f"Checkpoint tersimpan di: {Config.CHECKPOINT_PATH}")